[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jdsanch1/simrc/blob/master/02.%20Parte%202/13.%20Clase%2013/13Class%20NB.ipynb)
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/jdsanch1/SimRC/master?labpath=02.%20Parte%202%2F13.%20Clase%2013%2F13Class%20NB.ipynb)

In [ ]:
import importlib, subprocess, sys

_required = ["yfinance", "pandas", "numpy", "matplotlib", "seaborn", "scipy", "sklearn", "statsmodels", "cvxpy"]
_missing  = [pkg for pkg in _required if importlib.util.find_spec(pkg) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + _missing)

# Clase 13: Portafolio con bono — Monte Carlo vs Markowitz

**[Juan Diego Sánchez Torres](https://www.researchgate.net/profile/Juan_Diego_Sanchez_Torres)**  
*Profesor*, [MAF ITESO](http://maf.iteso.mx/web/general/detalle?group_id=5858156)  
dsanchez@iteso.mx

## Objetivos de aprendizaje

- Comparar **tres métodos de simulación** (normal, histograma, KDE) para modelar rendimientos futuros.
- Clasificar los precios finales simulados en **regiones de riesgo** definidas por strikes.
- Construir portafolios con **bono** usando estimadores robustos y CVXPY.
- Superponer la frontera eficiente de **Markowitz** sobre la nube de Monte Carlo con bono.
- Entender el **precio sombra del riesgo** como el Sharpe del portafolio tangente (Boyd §5.6).

---

## Introducción teórica

### Tres enfoques de simulación

En las Clases 7–8 introdujimos tres formas de generar rendimientos para simulación:

| Modelo | Genera rendimientos como | Captura colas pesadas | Continuo |
|--------|--------------------------|:---------------------:|:--------:|
| **Normal** | $r_t \sim \mathcal{N}(\hat{\mu}, \hat{\sigma}^2)$ | No | Sí |
| **Histograma** | Muestreo de distribución empírica discreta | Sí | No |
| **KDE** | Muestreo de kernel suavizado | Sí | Sí |

### Regiones de riesgo

Dados dos niveles de precio $K_1 < K_2$, clasificamos el precio final $S_T$ simulado en tres regiones:
- $S_T < K_1$ — zona de **pérdida** significativa
- $K_1 \leq S_T \leq K_2$ — zona **neutral**
- $S_T > K_2$ — zona de **ganancia** significativa

La distribución de frecuencias entre regiones depende del modelo de simulación y revela las diferencias en colas.

## 1. Datos y estimadores robustos

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import cvxpy as cp
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import sklearn.covariance as skcov
import statsmodels.api as sm
from sklearn.neighbors import KernelDensity

sns.set_theme(style="darkgrid", palette="tab10")
plt.rcParams.update({"figure.dpi": 100, "figure.figsize": (12, 5)})
pd.set_option("display.precision", 4)

In [ ]:
ticker = "AAPL"
data = yf.download(ticker, start="2025-01-01", end="2025-03-27",
                   auto_adjust=True, progress=False)
closes = data["Close"].dropna()
daily_returns = np.log(closes / closes.shift(1)).dropna()

# Estimadores robustos
huber = sm.robust.scale.Huber()
mu, sigma_h = huber(daily_returns.values.flatten())

S0 = closes.iloc[-1]
print(f"Precio actual: ${S0:.2f}")
print(f"μ Huber: {mu:.6f}, σ Huber: {sigma_h:.6f}")

---
## 2. Simulación Monte Carlo — Modelo Normal

Bajo el supuesto GBM, generamos $N$ trayectorias de $T$ días:

$$
S_T^{(i)} = S_0 \exp\!\left(\sum_{t=1}^{T} r_t^{(i)}\right), \qquad r_t^{(i)} \sim \mathcal{N}(\hat{\mu}, \hat{\sigma}^2)
$$

In [ ]:
np.random.seed(42)
T = 360
N = 10000
K1, K2 = 160, 220  # Fronteras de las regiones

# Modelo normal
sim_r_normal = np.random.normal(mu, sigma_h, size=(T, N))
ST_normal = S0 * np.exp(np.cumsum(sim_r_normal, axis=0))[-1]

# Clasificar en regiones
def classify(ST, K1, K2):
    below = (ST < K1).sum()
    between = ((ST >= K1) & (ST <= K2)).sum()
    above = (ST > K2).sum()
    total = len(ST)
    return {"S_T < K₁": below/total, "K₁ ≤ S_T ≤ K₂": between/total, "S_T > K₂": above/total}

freq_normal = classify(ST_normal, K1, K2)
print(f"Modelo Normal — Distribución de S_T en {N:,} trayectorias:")
for region, pct in freq_normal.items():
    print(f"  {region}: {pct:.1%}")

---
## 3. Simulación Monte Carlo — Histograma empírico

In [ ]:
values, bin_edges = np.histogram(daily_returns, bins=250)
weights = values / values.sum()
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

sim_r_hist = np.random.choice(bin_centers, size=(T, N), p=weights)
ST_hist = S0 * np.exp(np.cumsum(sim_r_hist, axis=0))[-1]

freq_hist = classify(ST_hist, K1, K2)
print(f"Modelo Histograma — Distribución de S_T:")
for region, pct in freq_hist.items():
    print(f"  {region}: {pct:.1%}")

---
## 4. Simulación Monte Carlo — KDE

In [ ]:
kde = KernelDensity(kernel="gaussian", bandwidth=0.002).fit(
    daily_returns.values.reshape(-1, 1))
sim_r_kde = kde.sample(n_samples=T * N, random_state=42).reshape(T, N)
ST_kde = S0 * np.exp(np.cumsum(sim_r_kde, axis=0))[-1]

freq_kde = classify(ST_kde, K1, K2)
print(f"Modelo KDE — Distribución de S_T:")
for region, pct in freq_kde.items():
    print(f"  {region}: {pct:.1%}")

---
## 5. Comparación de los tres modelos

In [ ]:
# Tabla comparativa
comp = pd.DataFrame({
    "Normal": freq_normal,
    "Histograma": freq_hist,
    "KDE": freq_kde,
})
comp

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ["tab:red", "tab:gray", "tab:green"]

for ax, (model, ST, title) in zip(axes, [
    ("Normal", ST_normal, "Normal"),
    ("Histograma", ST_hist, "Histograma"),
    ("KDE", ST_kde, "KDE"),
]):
    sns.histplot(ST, bins=80, stat="density", ax=ax, alpha=0.6, color="steelblue")
    ax.axvline(K1, color="red", linestyle="--", label=f"K₁ = {K1}")
    ax.axvline(K2, color="green", linestyle="--", label=f"K₂ = {K2}")
    ax.axvline(S0, color="black", linestyle=":", label=f"S₀ = {S0:.0f}")
    ax.set_title(f"Modelo {title}")
    ax.set_xlabel("Precio final S_T")
    ax.legend(fontsize=8)

plt.suptitle(f"Distribución de S_T a {T} días ({N:,} trayectorias)", fontsize=14)
plt.tight_layout()

In [ ]:
# Gráfico de barras comparativo
comp_t = comp.T
comp_t.plot(kind="bar", figsize=(10, 5), color=colors)
plt.title("Frecuencia por región — Comparación de modelos")
plt.ylabel("Proporción")
plt.xticks(rotation=0)
plt.legend(title="Región")
plt.tight_layout()

---
## 6. Portafolio con bono — Monte Carlo vs Markowitz

Ahora extendemos el análisis a un **portafolio multi-activo con bono**, comparando la simulación MC con la frontera eficiente de Markowitz (CVXPY).

In [ ]:
# Portafolio multi-activo
tickers_port = ["AAPL", "AMZN", "MSFT", "KO", "AA"]
data_port = yf.download(tickers_port, start="2025-01-01", end="2025-03-27",
                        auto_adjust=True, progress=False)
closes_port = data_port["Close"].dropna()
returns_port = np.log(closes_port / closes_port.shift(1)).dropna()

# Estimadores robustos
mu_port = np.array([huber(returns_port[col].values)[0] for col in returns_port.columns])
Sigma_port = skcov.LedoitWolf().fit(returns_port).covariance_

# Extender con bono
n_stocks = len(tickers_port)
r_f = 0.045 / 252
mu_ext = np.append(mu_port, r_f)
Sigma_ext = np.zeros((n_stocks + 1, n_stocks + 1))
Sigma_ext[:n_stocks, :n_stocks] = Sigma_port
labels_ext = tickers_port + ["BOND"]

print(f"Activos: {labels_ext}")

In [ ]:
# MC con bono: 25,000 portafolios
np.random.seed(42)
n_mc = 25000
w_mc = np.random.dirichlet(np.ones(n_stocks + 1), n_mc)
ret_mc = 252 * w_mc @ mu_ext
risk_mc = np.array([np.sqrt(252 * w @ Sigma_ext @ w) for w in w_mc])
sharpe_mc = np.where(risk_mc > 0, ret_mc / risk_mc, 0)

# Frontera eficiente con bono (CVXPY)
def efficient_frontier(mu_vec, Sigma, n_points=200):
    n = len(mu_vec)
    w = cp.Variable(n)
    target = cp.Parameter()
    risk = cp.quad_form(w, Sigma)
    prob = cp.Problem(cp.Minimize(risk),
                      [mu_vec @ w == target, cp.sum(w) == 1, w >= 0])
    means, stds = [], []
    for r in np.linspace(mu_vec.min(), mu_vec.max(), n_points):
        target.value = r
        prob.solve(solver=cp.ECOS, warm_start=True)
        if prob.status == "optimal":
            means.append(252 * r)
            stds.append(np.sqrt(252 * risk.value))
    return np.array(means), np.array(stds)

ef_means, ef_stds = efficient_frontier(mu_ext, Sigma_ext)
print(f"MC: {n_mc:,} portafolios | Frontera: {len(ef_means)} puntos")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

# Nube MC
sc = ax.scatter(risk_mc, ret_mc, c=sharpe_mc, cmap="RdYlBu", s=3, alpha=0.3)
plt.colorbar(sc, label="Sharpe (MC)")

# Frontera eficiente
ax.plot(ef_stds, ef_means, "b-", linewidth=2.5, label="Frontera eficiente (CVXPY)")

# Max Sharpe MC
idx_best = sharpe_mc.argmax()
ax.scatter(risk_mc[idx_best], ret_mc[idx_best], marker="*", s=300,
           color="red", zorder=5, edgecolors="black",
           label=f"Max Sharpe MC (S={sharpe_mc[idx_best]:.3f})")

ax.set_xlabel("Riesgo (σ anualizada)")
ax.set_ylabel("Rendimiento esperado (anualizado)")
ax.set_title("Portafolio con bono: Monte Carlo vs. Markowitz (CVXPY)")
ax.legend(loc="upper left")
plt.tight_layout()

### Interpretación

La **frontera eficiente** (línea azul) envuelve la nube MC y se extiende hasta el origen — reflejando que el bono permite reducir el riesgo a cero asignando el 100% al bono. La optimización de Markowitz encuentra la solución **exacta** que Monte Carlo solo aproxima.

---

## Navegación del curso

← **Anterior**: Clase 12: Optimización avanzada  
→ **Siguiente**: Clase 14: Opciones barrera

---

## 7. Referencias bibliográficas

- **Boyd, S. & Vandenberghe, L.** (2004). *Convex Optimization*. Cambridge University Press. — §4.3.2 (SOCP/CML), §5.6 (precios sombra).
- **Glasserman, P.** (2003). *Monte Carlo Methods in Financial Engineering*. Springer.
- **Luenberger, D. G.** (2013). *Investment Science* (2nd ed.). Oxford University Press. — Cap. 6–8.
- **Markowitz, H.** (1952). Portfolio Selection. *The Journal of Finance*, 7(1), 77–91.
- **Sharpe, W. F.** (1964). Capital Asset Prices. *The Journal of Finance*, 19(3), 425–442.
- **Tobin, J.** (1958). Liquidity Preference as Behavior Towards Risk. *The Review of Economic Studies*, 25(2), 65–86.
- **Tsay, R. S.** (2010). *Analysis of Financial Time Series* (3rd ed.). Wiley.
- **Venegas Martínez, F.** (2008). *Riesgos financieros y económicos* (2a ed.). Cengage Learning.